In [ ]:
import sys
# from pathlib import Path
# Ensure src/ is in path (same convention as notebooks)
# sys.path.insert(0, str(Path(__file__).resolve().parent.parent))

sys.path.insert(0, '../src')

import numpy as np
from bands.base import HamiltonianModel
from bands.graphene import (
    SingleLayerGrapheneTB,
    BilayerGrapheneTB,
    SingleLayerGrapheneKP,
    BilayerGrapheneKP,
    BilayerGrapheneKPAA,
)


In [ ]:
"""
验证 K 点 Dirac 速度。

对每种模型：
1. 在 K 点计算解析速度算符 v_α(k)
2. 变换到本征基：v^α_{mn} = V† v_α V
3. 验证 Dirac 能带对应的对角元 ≈ ±vF
4. 对比解析 vs 有限差分版本
"""
results = []

# ── 单层 TB ──────────────────────────────────────────
tb = SingleLayerGrapheneTB(t=2.78)
k_K = tb.high_symmetry_points()['K']
E_K, V_K = tb.solve(k_K)

vx, vy = tb.velocity_operator(k_K)
vx_eig = V_K.conj().T @ vx @ V_K
vy_eig = V_K.conj().T @ vy @ V_K

# 在 K 点，本征值为 0 (简并)，速度矩阵应给出 Fermi 速度
vF_tb = np.sqrt(3) / 2 * 2.78  # ≈ 2.408
v_abs = np.sqrt(np.abs(vx_eig[0, 1])**2 + np.abs(vy_eig[0, 1])**2)
results.append((
    f"SL TB vF at K: {v_abs:.4f} (expect {vF_tb:.4f})",
    np.allclose(v_abs, vF_tb, atol=1e-3)
))

# 解析 vs 有限差分对比 (K 点附近)
k_near = k_K + np.array([0.01, 0.0])
vx_an, vy_an = tb.velocity_operator(k_near)
# 调用基类的有限差分实现
vx_fd, vy_fd = HamiltonianModel.velocity_operator(tb, k_near)

print(vx_fd.real, vy_fd.real, vx_an.real, vy_an.real)
print(np.linalg.eigvalsh(vx_fd))

diff_v = np.max(np.abs(vx_an - vx_fd)) + np.max(np.abs(vy_an - vy_fd))
results.append((
    f"SL TB analytical vs FD diff: {diff_v:.2e}",
    diff_v < 1e-6
))


> `D:\work\plasmon-polariton\08-notebooks\Gr-tb-nn-dos-velocity+nnn.ipynb`

In [ ]:
hbar = 1.055e-34 # J · s
e = 1.602e-19
one_Joule_in_eV = 1 / e

factor = 2.46 / one_Joule_in_eV / hbar / 1e10 # from (eV) to (m/s)
print(f'{factor:.4g}')

In [ ]:
print(f"{2.40057056*factor:.4g}") # m/s

In [ ]:

# ── 单层 KP ──────────────────────────────────────────
# 注意：K 点二重简并，本征基任意，需偏移到非简并 k 点测 vF
kp = SingleLayerGrapheneKP(t=2.78, valley=+1)
k_near_kp = kp.K_point + np.array([0.001, 0.0])
E_kp_near, V_kp_near = kp.solve(k_near_kp)
vx_kp, vy_kp = kp.velocity_operator(k_near_kp)
vx_eig_kp = V_kp_near.conj().T @ vx_kp @ V_kp_near
vy_eig_kp = V_kp_near.conj().T @ vy_kp @ V_kp_near

vF_kp = kp.vF
# 非简并时 |v^α_{01}| = vF
v_abs_kp = np.sqrt(np.abs(vx_eig_kp[0, 1])**2 + np.abs(vy_eig_kp[0, 1])**2)

results.append((
    f"SL KP vF near K: {v_abs_kp:.4f} (expect {vF_kp:.4f})",
    np.allclose(v_abs_kp, vF_kp, atol=1e-3)
))

# KP vs FD
vx_kp_fd, vy_kp_fd = HamiltonianModel.velocity_operator(kp, k_near)

print(vx_kp.real, vy_kp.real,vx_kp_fd.real, vy_kp_fd.real, vF_kp, v_abs_kp)
print()
print(np.linalg.eigvalsh(vx_kp_fd))
print(np.linalg.eigvalsh(vx_kp))


diff_kp = np.max(np.abs(vx_kp - vx_kp_fd)) + np.max(np.abs(vy_kp - vy_kp_fd))
results.append((
    f"SL KP analytical vs FD diff: {diff_kp:.2e}",
    diff_kp < 1e-6
))


In [ ]:
print(f"{2.40755062*factor:.4g}") # m/s

In [ ]:

# ── 双层 KP (AB) ─────────────────────────────────────
kp_bl = BilayerGrapheneKP(t=2.78, gamma1=0.39, valley=+1)
vx_bl, vy_bl = kp_bl.velocity_operator(k_K)
vx_bl_fd, vy_bl_fd = HamiltonianModel.velocity_operator(kp_bl, k_K)

print(vx_bl.real, vy_bl.real,vx_bl_fd.real, vy_bl_fd.real)
print()
print(np.linalg.eigvalsh(vx_bl_fd))
print(np.linalg.eigvalsh(vx_bl))

diff_bl = np.max(np.abs(vx_bl - vx_bl_fd)) + np.max(np.abs(vy_bl - vy_bl_fd))
results.append((
    f"BL KP analytical vs FD diff: {diff_bl:.2e}",
    diff_bl < 1e-6
))


In [ ]:

# ── 双层 KP (AA) ─────────────────────────────────────
kp_aa = BilayerGrapheneKPAA(t=2.78, gamma_aa=0.2, valley=+1)
vx_aa, vy_aa = kp_aa.velocity_operator(k_K)
vx_aa_fd, vy_aa_fd = HamiltonianModel.velocity_operator(kp_aa, k_K)

print(vx_aa.real, vy_aa.real,vx_aa_fd.real, vy_aa_fd.real)
print()
print(np.linalg.eigvalsh(vx_aa_fd))
print(np.linalg.eigvalsh(vx_aa))


diff_aa = np.max(np.abs(vx_aa - vx_aa_fd)) + np.max(np.abs(vy_aa - vy_aa_fd))
results.append((
    f"AA KP analytical vs FD diff: {diff_aa:.2e}",
    diff_aa < 1e-6
))


In [ ]:

# ── 双层 TB ──────────────────────────────────────────
tb_bl = BilayerGrapheneTB(t=2.78, gamma1=0.39)
vx_tb_bl, vy_tb_bl = tb_bl.velocity_operator(k_near)
vx_tb_bl_fd, vy_tb_bl_fd = HamiltonianModel.velocity_operator(tb_bl, k_near)

print(vx_tb_bl.real, vy_tb_bl.real,vx_tb_bl_fd.real, vy_tb_bl_fd.real)
print()
print(np.linalg.eigvalsh(vx_tb_bl_fd))
print(np.linalg.eigvalsh(vx_tb_bl))

diff_tb_bl = np.max(np.abs(vx_tb_bl - vx_tb_bl_fd)) + np.max(np.abs(vy_tb_bl - vy_tb_bl_fd))
results.append((
    f"BL TB analytical vs FD diff: {diff_tb_bl:.2e}",
    diff_tb_bl < 1e-6
))


In [ ]:

# ── TBG BM ───────────────────────────────────────────
try:
    from bands.tbg_bm import BistritzMacDonaldTBG
    bm = BistritzMacDonaldTBG(theta=3.0, n_shells=2)
    k_moire_K = bm.high_symmetry_points()['K']
    vx_bm, vy_bm = bm.velocity_operator(k_moire_K)
    vx_bm_fd, vy_bm_fd = HamiltonianModel.velocity_operator(bm, k_moire_K)

    print(vx_bm.real, vy_bm.real,vx_bm_fd.real, vy_bm_fd.real)
    print()
    print(np.linalg.eigvalsh(vx_bm_fd))
    print(np.linalg.eigvalsh(vx_bm))

    diff_bm = np.max(np.abs(vx_bm - vx_bm_fd)) + np.max(np.abs(vy_bm - vy_bm_fd))
    results.append((
        f"BM (theta=3 deg) analytical vs FD diff: {diff_bm:.2e}",
        diff_bm < 1e-5
    ))

    # 检查 BM 速度矩阵的 Dirac 结构
    Nq = bm.Nq
    # 对第一个 Q 点 (top layer)，提取 2×2 速度块
    vx_block = vx_bm[0:2, 0:2]
    expected_vx_00 = -bm.vF * bm.xi  # -vF * ξ (∂H/∂k_x 的 [0,1] 元实部)
    results.append((
        f"BM top-layer vx block ~ Dirac: max|vx - expected| = {np.max(np.abs(vx_block - expected_vx_00 * np.array([[0, 1], [1, 0]]))):.2e}",
        True  # informational
    ))
except Exception as e:
    results.append((f"BM model: {e}", False))


In [ ]:
print(f"{5.92460335 / one_Joule_in_eV / hbar / 1e10:.4g}")
# factor = 2.46 / one_Joule_in_eV / hbar / 1e10 # from (eV) to (m/s)


In [ ]:

# ── 输出 ─────────────────────────────────────────────
if True:
    print("\n" + "=" * 52)
    print("  Dirac Velocity Verification")
    print("=" * 52)
    passed = 0
    for desc, ok in results:
        status = "PASS" if ok else ("INFO" if ok is True else "FAIL!")
        print(f"  [{status}] {desc}")
        if ok:
            passed += 1
    print("-" * 52)
    print(f"  Passed: {passed}/{len(results)}")
    print("=" * 52)


在石墨烯/狄拉克费米子体系常用的“狄拉克单位”（把普朗克常数除以2π ħ 与费米速度 v_F 一起用于能量—长度换算）中，能量 E 与长度 L 的关系为

# E = ħ v_F / L

单位 \[eV\] = (eV s) / s = (J s) (eV/J) (m/s) / (angstrom*(m/angstrom))

## L: 晶格常数

In [ ]:
from scipy.constants import eV, hbar, angstrom

v_F = 9e5 # m/s

# 1 eV = 1.602176634e-19 J
# 1 hbar = 1.0545718176461565e-34 J s
# 1 angstrom = 1e-10 m

hbar/eV * v_F / (2.46*angstrom)